In [3]:

import math, sys

nodes_minimax = 0
nodes_ab = 0

def initial_state():
    return [' '] * 9

def print_board(s):
    for r in range(3):
        print("".join(f" {c} " if c!=' ' else "   " for c in s[r*3:(r+1)*3]))
        if r < 2:
            print("---+---+---")

def actions(s):
    return [i for i,v in enumerate(s) if v == ' ']

def result(s, a, p):
    if s[a] != ' ': raise ValueError("invalid action")
    ns = s.copy(); ns[a] = p; return ns

def winner(s):
    lines = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
    for a,b,c in lines:
        if s[a]==s[b]==s[c] and s[a] in ('X','O'): return s[a]
    if all(c!=' ' for c in s): return 'DRAW'
    return None

def utility(s):
    w = winner(s)
    if w == 'X': return 1
    if w == 'O': return -1
    if w == 'DRAW': return 0
    return None

def player(s):
    return 'X' if s.count('X') == s.count('O') else 'O'

def minimax_decision(s):
    cur = player(s); best = None
    if cur == 'X':
        bv = -math.inf
        for a in actions(s):
            v = min_value(result(s,a,'X'))
            if v > bv: bv, best = v, a
    else:
        bv = math.inf
        for a in actions(s):
            v = max_value(result(s,a,'O'))
            if v < bv: bv, best = v, a
    return best

def max_value(s):
    global nodes_minimax
    nodes_minimax += 1
    u = utility(s)
    if u is not None: return u
    v = -math.inf
    for a in actions(s):
        v = max(v, min_value(result(s,a,'X')))
    return v

def min_value(s):
    global nodes_minimax
    nodes_minimax += 1
    u = utility(s)
    if u is not None: return u
    v = math.inf
    for a in actions(s):
        v = min(v, max_value(result(s,a,'O')))
    return v

def alphabeta_decision(s):
    cur = player(s); best = None
    if cur == 'X':
        alpha, beta, bv = -math.inf, math.inf, -math.inf
        for a in actions(s):
            v = min_value_ab(result(s,a,'X'), alpha, beta)
            if v > bv: bv, best = v, a
            alpha = max(alpha, bv)
    else:
        alpha, beta, bv = -math.inf, math.inf, math.inf
        for a in actions(s):
            v = max_value_ab(result(s,a,'O'), alpha, beta)
            if v < bv: bv, best = v, a
            beta = min(beta, bv)
    return best

def max_value_ab(s, alpha, beta):
    global nodes_ab
    nodes_ab += 1
    u = utility(s)
    if u is not None: return u
    v = -math.inf
    for a in actions(s):
        v = max(v, min_value_ab(result(s,a,'X'), alpha, beta))
        if v >= beta: return v
        alpha = max(alpha, v)
    return v

def min_value_ab(s, alpha, beta):
    global nodes_ab
    nodes_ab += 1
    u = utility(s)
    if u is not None: return u
    v = math.inf
    for a in actions(s):
        v = min(v, max_value_ab(result(s,a,'O'), alpha, beta))
        if v <= alpha: return v
        beta = min(beta, v)
    return v

def status_str(s):
    w = winner(s)
    if w == 'X': return "X wins"
    if w == 'O': return "O wins"
    if w == 'DRAW': return "Draw"
    return "Continue"

def human_move(s):
    legal = actions(s)
    while True:
        v = input("Move (0-8) or q: ").strip()
        if v.lower() == 'q': sys.exit(0)
        try:
            m = int(v)
            if m in legal: return m
        except: pass
        print("Invalid. Legal:", legal)

def run_human(use_ab=False):
    global nodes_minimax, nodes_ab
    nodes_minimax = nodes_ab = 0
    s = initial_state()
    print("X (AI) goes first. You are O.")
    while True:
        cur = player(s)
        if cur == 'X':
            a = alphabeta_decision(s) if use_ab else minimax_decision(s)
            s = result(s,a,'X')
            print_board(s); print(f"Move: {a}"); print("Status:", status_str(s))
        else:
            m = human_move(s); s = result(s,m,'O')
            print_board(s); print("Status:", status_str(s))
        if winner(s) is not None:
            print("\nFinal:")
            print_board(s)
            print("Result:", status_str(s))
            print("Minimax nodes:", nodes_minimax, "Alpha-Beta nodes:", nodes_ab)
            print("Note: Alpha-Beta prunes branches to reduce nodes.")
            break

def run_ai_vs_ai(x_minimax=True, o_ab=True):
    global nodes_minimax, nodes_ab
    nodes_minimax = nodes_ab = 0
    s = initial_state()
    while True:
        cur = player(s)
        if cur == 'X':
            a = minimax_decision(s) if x_minimax else alphabeta_decision(s)
            s = result(s,a,'X')
            print_board(s); print(f"Move: {a}"); print("Status:", status_str(s))
        else:
            a = alphabeta_decision(s) if o_ab else minimax_decision(s)
            s = result(s,a,'O')
            print_board(s); print(f"Move: {a}"); print("Status:", status_str(s))
        if winner(s) is not None:
            print("\nFinal:")
            print_board(s)
            print("Result:", status_str(s))
            print("Minimax nodes:", nodes_minimax, "Alpha-Beta nodes:", nodes_ab)
            print("Note: Alpha-Beta prunes branches to reduce nodes.")
            break

def main():
    print("1) Human vs AI (Minimax)  2) Human vs AI (Alpha-Beta)  3) AI vs AI (X=Minimax,O=AB)  q) Quit")
    c = input("Choice: ").strip().lower()
    if c == '1': run_human(use_ab=False)
    elif c == '2': run_human(use_ab=True)
    elif c == '3': run_ai_vs_ai()
    else: print("Exit")

if __name__ == "__main__":
    main()

1) Human vs AI (Minimax)  2) Human vs AI (Alpha-Beta)  3) AI vs AI (X=Minimax,O=AB)  q) Quit


Choice:  3


 X       
---+---+---
         
---+---+---
         
Move: 0
Status: Continue
 X       
---+---+---
    O    
---+---+---
         
Move: 4
Status: Continue
 X  X    
---+---+---
    O    
---+---+---
         
Move: 1
Status: Continue
 X  X  O 
---+---+---
    O    
---+---+---
         
Move: 2
Status: Continue
 X  X  O 
---+---+---
    O    
---+---+---
 X       
Move: 6
Status: Continue
 X  X  O 
---+---+---
 O  O    
---+---+---
 X       
Move: 3
Status: Continue
 X  X  O 
---+---+---
 O  O  X 
---+---+---
 X       
Move: 5
Status: Continue
 X  X  O 
---+---+---
 O  O  X 
---+---+---
 X  O    
Move: 7
Status: Continue
 X  X  O 
---+---+---
 O  O  X 
---+---+---
 X  O  X 
Move: 8
Status: Draw

Final:
 X  X  O 
---+---+---
 O  O  X 
---+---+---
 X  O  X 
Result: Draw
Minimax nodes: 557487 Alpha-Beta nodes: 2431
Note: Alpha-Beta prunes branches to reduce nodes.
